# Exploratory Data Analysis — Online Retail II

Analyses revenue trends, month-over-month growth, and geographic distribution.

**Dataset:** [Online Retail II](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) — place the file at `data/online_retail_II.xlsx`

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_excel("data/online_retail_II.xlsx")
print(df.shape)

## 2. Basic Exploration

In [ ]:
# Extract month from invoice date for time-series analysis
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['Revenue'] = df['Quantity'] * df['Price']

print(df.describe())
print("\nMissing values:")
print(df.isna().sum())

## 3. Monthly Revenue & MoM Growth

In [ ]:
monthly_revenue = df.groupby('Month').agg({
    'Invoice': 'nunique',
    'Customer ID': 'nunique',
    'Revenue': 'sum'
}).reset_index()

# Month-over-month growth rate
monthly_revenue['MOM_growth'] = monthly_revenue['Revenue'].pct_change() * 100
monthly_revenue

In [ ]:
# Green bars = growth, red bars = decline
plt.figure(figsize=(12, 5))
plt.bar(
    monthly_revenue['Month'].astype(str),
    monthly_revenue['MOM_growth'],
    color=np.where(monthly_revenue['MOM_growth'] <= 0, 'red', 'green')
)
plt.title('Month-over-Month Revenue Growth (%)')
plt.xlabel('Month')
plt.ylabel('MoM Growth (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Top 10 Countries by Revenue

In [ ]:
country_revenue = df.groupby('Country').agg({
    'Customer ID': 'nunique',
    'Revenue': 'sum'
}).sort_values('Revenue', ascending=False).reset_index()

top_10 = country_revenue[:10]
top_10['Revenue_share'] = top_10['Revenue'] / country_revenue['Revenue'].sum() * 100
top_10

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_10['Country'], top_10['Revenue'])
plt.gca().invert_yaxis()
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.xticks(ticks=[0, 2e6, 4e6, 6e6, 8e6],
           labels=['£0', '£2M', '£4M', '£6M', '£8M'])
plt.tight_layout()
plt.show()

In [ ]:
# Revenue share pie chart (top 10 countries)
plt.pie(top_10['Revenue_share'], labels=top_10['Country'], autopct='%1.1f%%')
plt.title('Revenue Share by Country')
plt.show()

## 5. Monthly Revenue by Country

In [ ]:
country_monthly = df.groupby(['Month', 'Country']).agg({
    'Invoice': 'nunique',
    'Quantity': 'sum',
    'Customer ID': 'nunique',
    'Revenue': 'sum'
}).reset_index()

# Sort so pct_change() is applied correctly within each country
country_monthly = country_monthly.sort_values(['Country', 'Month'])
country_monthly['MOM_growth'] = country_monthly.groupby('Country')['Revenue'].pct_change() * 100
country_monthly